In [ ]:
import numpy as np
import os
import scipy.io
import matplotlib.pyplot as plt

## 1. Load SC and check symmetry

In [ ]:
### Infer the AAL-90 parcel order of the SC / time series
def infer_aal90_order(SC, verbose=True):
    """
    Infer the AAL-90 parcel order of a 90x90 SC (or FC / Ceff) matrix.

    Three candidate orders:
    - 'non-symmetric' : L1..L45, R1..R45       (LLRR)
    - 'mirrored'      : L1..L45, R45..R1       (right hemisphere reversed;
                        the order of the sleep W/N1/N2/N3 and COMA AAL90 data)
    - 'symmetric'     : L1, R1, L2, R2, ...    (SPM interleaved LRLR)

    Method: for each candidate, pair every left region with its homotopic
    right region and correlate the LL and RR blocks. In the correct order
    the two hemispheres are near copies (r ~ 0.9-1.0); wrong orders give
    r ~ 0.1-0.2.

    NOTE (2026-09-28): the previous version only compared intra- vs
    inter-hemispheric SC density. That test cannot distinguish LLRR from
    mirrored (both put the left hemisphere first), so it returned
    'non-symmetric' for mirrored data and mislabelled 44/45 right-hemisphere
    parcels.

    Returns 'non-symmetric', 'mirrored' or 'symmetric'.
    """
    if SC.shape != (90, 90):
        raise ValueError("SC matrix must be 90x90 for AAL-90 parcellation.")
    A = np.abs(SC) / np.abs(SC).max()
    k = np.arange(45)
    candidates = {
        'non-symmetric': (k, k + 45),
        'mirrored': (k, 89 - k),
        'symmetric': (2 * k, 2 * k + 1),
    }
    iu = np.triu_indices(45, 1)
    scores = {}
    for name, (L, R) in candidates.items():
        scores[name] = np.corrcoef(A[np.ix_(L, L)][iu], A[np.ix_(R, R)][iu])[0, 1]
    best = max(scores, key=scores.get)
    if verbose:
        for name, r in scores.items():
            print(f"LL~RR block similarity, {name:13s}: r = {r:.3f}")
        print(f"=> Parcel order: {best}")
    ranked = sorted(scores.values(), reverse=True)
    if ranked[0] < 0.5 or ranked[0] - ranked[1] < 0.2:
        print("[WARN] parcel order is ambiguous -- check it manually")
    return best


def check_aal90_mirrored(labels, M=None, name='SC'):
    """
    Verify the mirrored AAL-90 order L1..L45, R45..R1 of the sleep
    W/N1/N2/N3 and COMA data sets; raise ValueError if it does not hold.

    - labels: the 90 labels attached to the data columns. Column c < 45 must
      be a left region and column 89-c its right homologue.
    - M: optional 90x90 SC / FC / Ceff matrix; infer_aal90_order(M) must
      return 'mirrored'.
    """
    labels = list(labels)
    if len(labels) != 90:
        raise ValueError(f"Expected 90 AAL labels, got {len(labels)}.")
    bad = [c for c in range(45)
           if not (labels[c].endswith('_L') and labels[89 - c] == labels[c][:-2] + '_R')]
    if bad:
        c = bad[0]
        raise ValueError(f"AAL_labels not in mirrored order L1..L45, R45..R1: {len(bad)} bad pairs, "
                         f"e.g. col {c} = {labels[c]} vs col {89 - c} = {labels[89 - c]}")
    if M is not None:
        order = infer_aal90_order(M, verbose=False)
        if order != 'mirrored':
            raise ValueError(f"{name} is in '{order}' order, not mirrored (L1..L45, R45..R1).")
    print("AAL90 order check passed: L1..L45, R45..R1" + (f" ({name} + labels)" if M is not None else " (labels)"))
################################################################
################################################################


In [ ]:
### LOAD SC
## SC directory
SC_dir = './'
data = scipy.io.loadmat(f'{SC_dir}SC_AAL90.mat')
SC = np.array(data['SC']).copy()
NPARCELS = SC.shape[0]
del data
PARCELS_ORDER = infer_aal90_order(SC)
print(PARCELS_ORDER)

plt.imshow(SC, cmap='turbo')
plt.title("Structural Connectivity Matrix")
plt.colorbar()
plt.show()

# 2. AAL90 $\to$ YEO7 mapping

Each AAL90 region is assigned to the Yeo et al. (2011) 7-network component with the largest voxel overlap in MNI space (left and right homologues pooled). Source table: `aal90_labels_YEO7.csv` (built by `make_aal90_labels_YEO7.py`).

- Caudate, Putamen, Pallidum and Thalamus are **not assigned**: the Yeo 2011 atlas is cortex-only. They stay in region-level analyses but are excluded from network summaries (82 of 90 regions mapped).
- Hippocampus and Amygdala lie mostly outside the cortical Yeo mask, so they are assigned by convention (Default and Limbic).
- Mapping corrected 2026-11-12. The earlier hand-made mapping had errors (Frontal_Med_Orb missing, Parietal_Inf in two networks, 9 regions in a different network than the atlas overlap gives).


In [ ]:
# AAL90 -> Yeo7 (Yeo et al. 2011, 7 networks) by maximal voxel overlap in MNI space,
# L+R homologues pooled. Source: aal90_labels_YEO7.csv (corrected 2026-11-12).
# Subcortical nuclei (Caudate, Putamen, Pallidum, Thalamus) are not in the cortical
# Yeo atlas and are left unassigned; Hippocampus->Default, Amygdala->Limbic by convention.
YEO7_AAL90_GROUPS_LABELS = {
    "YEO_VIS": [
        "Calcarine_L", "Calcarine_R",
        "Cuneus_L", "Cuneus_R",
        "Lingual_L", "Lingual_R",
        "Occipital_Sup_L", "Occipital_Sup_R",
        "Occipital_Mid_L", "Occipital_Mid_R",
        "Occipital_Inf_L", "Occipital_Inf_R",
        "Fusiform_L", "Fusiform_R",
    ],
    "YEO_SMN": [
        "Precentral_L", "Precentral_R",
        "Rolandic_Oper_L", "Rolandic_Oper_R",
        "Supp_Motor_Area_L", "Supp_Motor_Area_R",
        "Postcentral_L", "Postcentral_R",
        "Paracentral_Lobule_L", "Paracentral_Lobule_R",
        "Heschl_L", "Heschl_R",
        "Temporal_Sup_L", "Temporal_Sup_R",
    ],
    "YEO_DAN": [
        "Parietal_Sup_L", "Parietal_Sup_R",
    ],
    "YEO_VAN": [
        "Insula_L", "Insula_R",
        "Cingulum_Mid_L", "Cingulum_Mid_R",
        "SupraMarginal_L", "SupraMarginal_R",
    ],
    "YEO_LIM": [
        "Frontal_Sup_Orb_L", "Frontal_Sup_Orb_R",
        "Olfactory_L", "Olfactory_R",
        "Rectus_L", "Rectus_R",
        "ParaHippocampal_L", "ParaHippocampal_R",
        "Amygdala_L", "Amygdala_R",
        "Temporal_Pole_Sup_L", "Temporal_Pole_Sup_R",
        "Temporal_Pole_Mid_L", "Temporal_Pole_Mid_R",
        "Temporal_Inf_L", "Temporal_Inf_R",
    ],
    "YEO_FPN": [
        "Frontal_Mid_L", "Frontal_Mid_R",
        "Frontal_Mid_Orb_L", "Frontal_Mid_Orb_R",
        "Frontal_Inf_Oper_L", "Frontal_Inf_Oper_R",
        "Frontal_Inf_Tri_L", "Frontal_Inf_Tri_R",
        "Parietal_Inf_L", "Parietal_Inf_R",
    ],
    "YEO_DMN": [
        "Frontal_Sup_L", "Frontal_Sup_R",
        "Frontal_Inf_Orb_L", "Frontal_Inf_Orb_R",
        "Frontal_Sup_Medial_L", "Frontal_Sup_Medial_R",
        "Frontal_Med_Orb_L", "Frontal_Med_Orb_R",
        "Cingulum_Ant_L", "Cingulum_Ant_R",
        "Cingulum_Post_L", "Cingulum_Post_R",
        "Hippocampus_L", "Hippocampus_R",
        "Angular_L", "Angular_R",
        "Precuneus_L", "Precuneus_R",
        "Temporal_Mid_L", "Temporal_Mid_R",
    ],
}


### Non-symmetric AAL-90 Order (L then R)
AAL_labels_non_symmetric = [
    'Precentral_L', 'Frontal_Sup_L', 'Frontal_Sup_Orb_L', 'Frontal_Mid_L', 'Frontal_Mid_Orb_L',
    'Frontal_Inf_Oper_L', 'Frontal_Inf_Tri_L', 'Frontal_Inf_Orb_L', 'Rolandic_Oper_L', 'Supp_Motor_Area_L',
    'Olfactory_L', 'Frontal_Sup_Medial_L', 'Frontal_Med_Orb_L', 'Rectus_L', 'Insula_L',
    'Cingulum_Ant_L', 'Cingulum_Mid_L', 'Cingulum_Post_L', 'Hippocampus_L', 'ParaHippocampal_L',
    'Amygdala_L', 'Calcarine_L', 'Cuneus_L', 'Lingual_L', 'Occipital_Sup_L',
    'Occipital_Mid_L', 'Occipital_Inf_L', 'Fusiform_L', 'Postcentral_L', 'Parietal_Sup_L',
    'Parietal_Inf_L', 'SupraMarginal_L', 'Angular_L', 'Precuneus_L', 'Paracentral_Lobule_L',
    'Caudate_L', 'Putamen_L', 'Pallidum_L', 'Thalamus_L', 'Heschl_L',
    'Temporal_Sup_L', 'Temporal_Pole_Sup_L', 'Temporal_Mid_L', 'Temporal_Pole_Mid_L', 'Temporal_Inf_L',

    'Precentral_R', 'Frontal_Sup_R', 'Frontal_Sup_Orb_R', 'Frontal_Mid_R', 'Frontal_Mid_Orb_R',
    'Frontal_Inf_Oper_R', 'Frontal_Inf_Tri_R', 'Frontal_Inf_Orb_R', 'Rolandic_Oper_R', 'Supp_Motor_Area_R',
    'Olfactory_R', 'Frontal_Sup_Medial_R', 'Frontal_Med_Orb_R', 'Rectus_R', 'Insula_R',
    'Cingulum_Ant_R', 'Cingulum_Mid_R', 'Cingulum_Post_R', 'Hippocampus_R', 'ParaHippocampal_R',
    'Amygdala_R', 'Calcarine_R', 'Cuneus_R', 'Lingual_R', 'Occipital_Sup_R',
    'Occipital_Mid_R', 'Occipital_Inf_R', 'Fusiform_R', 'Postcentral_R', 'Parietal_Sup_R',
    'Parietal_Inf_R', 'SupraMarginal_R', 'Angular_R', 'Precuneus_R', 'Paracentral_Lobule_R',
    'Caudate_R', 'Putamen_R', 'Pallidum_R', 'Thalamus_R', 'Heschl_R',
    'Temporal_Sup_R', 'Temporal_Pole_Sup_R', 'Temporal_Mid_R', 'Temporal_Pole_Mid_R', 'Temporal_Inf_R'
]

### Symmetric AAL-90 Order (Interleaved L-R, SPM order)
AAL_labels_symmetric = []
for i in range(45):
    AAL_labels_symmetric.append(AAL_labels_non_symmetric[i])
    AAL_labels_symmetric.append(AAL_labels_non_symmetric[i + 45])

### Mirrored AAL-90 Order (L1..L45, then R45..R1)
### This is the order of the sleep W/N1/N2/N3 and COMA AAL90 data sets.
AAL_labels_mirrored = AAL_labels_non_symmetric[:45] + AAL_labels_non_symmetric[45:][::-1]

print('Parcels order is:',PARCELS_ORDER)
if PARCELS_ORDER == 'non-symmetric':
    AAL_labels = AAL_labels_non_symmetric.copy()
elif PARCELS_ORDER == 'mirrored':
    AAL_labels = AAL_labels_mirrored.copy()
elif PARCELS_ORDER == 'symmetric':
    AAL_labels = AAL_labels_symmetric.copy()
else:
    raise ValueError(f"Unknown PARCELS_ORDER: {PARCELS_ORDER}")
check_aal90_mirrored(AAL_labels, SC)

In [ ]:
for i, region in enumerate(AAL_labels):
    print(i+1, region)

In [ ]:
# YEO7-to-index mapping
YEO7_indexes = {}
missing_by_YEO7 = {}

YEO7_indexes = {}
missing_by_YEO7 = {}

for YEO7, regions in YEO7_AAL90_GROUPS_LABELS.items():
    idx = [i for i, region in enumerate(AAL_labels) if region in regions]
    YEO7_indexes[YEO7] = idx

    # track missing labels (helpful debugging)
    present = set([AAL_labels[i] for i in idx])
    missing = [r for r in regions if r not in present]
    missing_by_YEO7[YEO7] = missing

# Optional: print warnings
for YEO7, missing in missing_by_YEO7.items():
    if len(missing) > 0:
        print(f"[WARN] YEO7 '{YEO7}' missing labels in AAL_labels: {missing}")

for YEO7, idx in YEO7_indexes.items():
    if len(idx) == 0:
        print(f"[WARN] YEO7 '{YEO7}' has 0 parcels mapped!")



## 3. Test bar plot

In [ ]:
import itertools
np.random.seed(0)

In [ ]:
# random data sets for demonstration
data_set0 = np.random.rand(NPARCELS) + 0
data_set1 = np.random.rand(NPARCELS) + 1
data_set2 = np.random.rand(NPARCELS) + 2
data_set3 = np.random.rand(NPARCELS) + 3

label_set0 = 'Set 0'
label_set1 = 'Set 1'
label_set2 = 'Set 2'
label_set3 = 'Set 3'

# Compute mean values per YEO7
YEO7_data_set0 = {YEO7: np.mean(data_set0[indexes]) for YEO7, indexes in YEO7_indexes.items()}
YEO7_data_set1 = {YEO7: np.mean(data_set1[indexes]) for YEO7, indexes in YEO7_indexes.items()}
YEO7_data_set2 = {YEO7: np.mean(data_set2[indexes]) for YEO7, indexes in YEO7_indexes.items()}
YEO7_data_set3 = {YEO7: np.mean(data_set3[indexes]) for YEO7, indexes in YEO7_indexes.items()}

# Prepare data
YEO7s = list(YEO7_data_set0.keys())
x = np.arange(len(YEO7s))  # the label locations
width = 0.2  # width of the bars

# Values
values0 = [YEO7_data_set0[YEO7] for YEO7 in YEO7s]
values1 = [YEO7_data_set1[YEO7] for YEO7 in YEO7s]
values2 = [YEO7_data_set2[YEO7] for YEO7 in YEO7s]
values3 = [YEO7_data_set3[YEO7] for YEO7 in YEO7s]

# Plot
plt.figure(figsize=(12, 6))
plt.bar(x - 3*width/2, values0, width, alpha=0.75, label=f'{label_set0}')
plt.bar(x -   width/2, values1, width, alpha=0.75, label=f'{label_set1}')
plt.bar(x +   width/2, values2, width, alpha=0.75, label=f'{label_set2}')
plt.bar(x + 3*width/2, values3, width, alpha=0.75, label=f'{label_set2}')

# Labels and legend
plt.ylabel("Data")
plt.title("Data per YEO7 Network [group]")
plt.xticks(x, YEO7s, rotation=45)
plt.tight_layout()
plt.legend()
plt.show()